# Notebook Scrapping 

Auteur : Arsena Julien

L'algorithme de scrapping a été généré par IA (Gémini) et a été vérifié manuellement. 

Cet algorithme a été réalisé dans le cadre d'une étude stastistique (**voir Notebook 'Projet Linguistique Julien Arsena'**)

## 1. Library import

In [ ]:
# pip install selenium webdriver-manager


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

## 2. Algo scrapping

In [ ]:
# Configuration du navigateur en mode headless
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--disable-gpu')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

print("Lancement du navigateur en mode headless...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = "https://atlas.lisn.upsaclay.fr/liste.html"
driver.get(url)

print("Attente du chargement des données (JavaScript)...")

try:
    # Attente de l'apparition du premier élément du tableau
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.TAG_NAME, "td"))
    )
    
    print("Finalisation du chargement du tableau...")
    # Temporisation pour permettre l'injection complète des données par le script de la page
    time.sleep(5) 
    
    print("Données chargées. Extraction en cours...")
    
    html_complet = driver.page_source
    soup = BeautifulSoup(html_complet, 'html.parser')
    
    data = []
    table = soup.find('table')
    
    if table:
        rows = table.find_all('tr')
        
        for row in rows:
            # Recherche des balises 'td' et 'th' pour gérer d'éventuelles variations de formatage
            cols = row.find_all(['td', 'th'])
            
            if len(cols) >= 7:
                # Nettoyage des chaînes de caractères pour éviter les sauts de ligne dans le CSV
                id_lang = cols[0].text.strip().replace('\n', ' ')
                
                # Exclusion de la ligne d'en-tête
                if id_lang.upper() == "ID":
                    continue
                
                ville = cols[1].text.strip().replace('\n', ' ')
                pays = cols[2].text.strip().replace('\n', ' ')
                
                # Extraction et formatage de l'URL du fichier audio
                audio_url = None
                audio_tag = row.find('audio')
                if audio_tag:
                    audio_file = None
                    if audio_tag.has_attr('src'):
                        audio_file = audio_tag['src']
                    elif audio_tag.find('source') and audio_tag.find('source').has_attr('src'):
                        audio_file = audio_tag.find('source')['src']
                    
                    if audio_file:
                        if not audio_file.startswith('http'):
                            audio_url = "https://atlas.lisn.upsaclay.fr/" + audio_file
                        else:
                            audio_url = audio_file
                
                longitude = cols[4].text.strip().replace('\n', ' ')
                latitude = cols[5].text.strip().replace('\n', ' ')
                transcription = cols[6].text.strip().replace('\n', ' ')
                
                data.append({
                    "ID": id_lang,
                    "Ville": ville,
                    "Pays": pays,
                    "Longitude": longitude,
                    "Latitude": latitude,
                    "Audio_URL": audio_url,
                    "Transcription": transcription
                })

        # Sauvegarde des données
        df = pd.DataFrame(data)
        df.to_csv("atlas_donnees_scrapees.csv", index=False, encoding='utf-8')
        print(f"Succès : {len(df)} entrées ont été enregistrées dans le fichier CSV.")
    else:
        print("Erreur : Aucun tableau trouvé sur la page.")

except Exception as e:
    print(f"Une erreur est survenue lors de l'exécution : {e}")

finally:
    # Fermeture de l'instance du navigateur
    driver.quit()

Lancement du navigateur en mode headless...
Attente du chargement des données (JavaScript)...
Finalisation du chargement du tableau...
Données chargées. Extraction en cours...
Succès : 1041 entrées ont été enregistrées dans le fichier CSV.


## 3. Data

In [9]:
df = pd.read_csv("atlas_donnees_scrapees.csv", encoding='utf-8')
display(df)

,ID,Ville,Pays,Longitude,Latitude,Audio_URL,Transcription
0,#,Ville,Pays,Longitude,Latitude,NaN,Transcription
1,1,Pieris,ItaliaNord,13.448098,45.811774,https://atlas.lisn.upsaclay.fr/sounds/Pieris.mp3,"Un zorno, co èreli drìo custionar de chi fùsse..."
2,2,Caraman,France,1.7583,43.5311,https://atlas.lisn.upsaclay.fr/sounds/Caraman.mp3,La bisa e le sorelh se disputavan ; cadun asse...
3,3,Bois-de-Villers,Belgique,4.823933,50.39001,https://atlas.lisn.upsaclay.fr/sounds/Bois-de-...,"I gn-avéve mârgaye ètur li bîje èt l' solia, c..."
4,4,Meilly-sur-Rouvres,France,4.562,47.206,https://atlas.lisn.upsaclay.fr/sounds/Meilly-s...,Lai bise ai peûs le Sulô s'airgognínt. Als vou...
...,...,...,...,...,...,...,...
1036,1036,Zell im Wiesental,Suisse,7.859324,47.709136,https://atlas.lisn.upsaclay.fr/sounds/Zell_im_...,"De Nordwind un Sunne hän, wo grad en Wanderer ..."
1037,1037,zuanga,NCWF,164.397191,-20.666893,https://atlas.lisn.upsaclay.fr/sounds/zuanga.mp3,[Nu mâ pweru we nya fâdjama A mê Dree.] Li pe ...
1038,1038,Zunzgen,Suisse,7.806166,47.452855,https://atlas.lisn.upsaclay.fr/sounds/CH_Nordw...,Äimòl händ sìch dr Nòrdwìnd und d Sùnne gschdr...
1039,1039,Zuoz,Suisse,9.959371,46.602293,https://atlas.lisn.upsaclay.fr/sounds/CH_Nordw...,Il vent dal nord ed il sulagl — Ün bel di as h...
